# LangGraph Example 1: Simple Workflow

This notebook demonstrates basic LangGraph concepts integrated with Beast Mode's ReflectiveModule pattern.

## What You'll Learn
- StateGraph with TypedDict state schema
- Simple node functions (state → state)
- Linear workflow: Input → Process → Output
- ReflectiveModule integration for built-in Jaeger tracing

## Prerequisites
- Jaeger running at http://localhost:16686
- `pip install langgraph`

In [ ]:
# Imports
import os
import sys
from typing import TypedDict, Dict, Any, List
from datetime import datetime

# Add project root to path
sys.path.insert(0, os.path.abspath('../..'))

from langgraph.graph import StateGraph, END
from src.rm_ddd.core.unified_reflective_module import (
    ReflectiveModule, 
    ModuleCapability, 
    ModuleHealth, 
    ModuleStatus
)

print("✅ Imports successful")
print("📊 Jaeger UI: http://localhost:16686")

## Step 1: Define State Schema

LangGraph uses TypedDict to define the structure of state flowing through the workflow.

In [ ]:
class SimpleWorkflowState(TypedDict):
    """State that flows through the workflow"""
    input_text: str
    processed_text: str
    output_text: str
    timestamp: str
    step_count: int

print("✅ State schema defined")
print("   Fields: input_text, processed_text, output_text, timestamp, step_count")

## Step 2: Create Workflow Orchestrator

Extends ReflectiveModule to get:
- Jaeger tracing via `trace_operation()`
- Prometheus metrics
- Redis auto-registration
- Health monitoring

In [ ]:
class SimpleWorkflowOrchestrator(ReflectiveModule):
    """Simple LangGraph workflow with ReflectiveModule integration"""

    def __init__(self):
        super().__init__()
        self.module_id = "SimpleWorkflowOrchestrator"
        self._logger.info("🚀 Initializing Simple Workflow Orchestrator")
        self.graph = self._create_graph()

    def get_module_info(self) -> Dict[str, Any]:
        return {
            "module_name": "SimpleWorkflowOrchestrator",
            "version": "1.0.0",
            "description": "Simple LangGraph workflow example",
            "graph_nodes": ["input_node", "process_node", "output_node"]
        }

    def get_capabilities(self) -> List[ModuleCapability]:
        return [
            ModuleCapability.CORE_FUNCTIONALITY,
            ModuleCapability.DATA_PROCESSING
        ]

    def get_health_status(self) -> ModuleHealth:
        return ModuleHealth(
            module_id=self.module_id,
            status=ModuleStatus.HEALTHY,
            health_score=1.0,
            issues=[],
            last_check=datetime.now()
        )

    def graceful_degradation(self, error: Exception) -> Dict[str, Any]:
        return {
            "degradation_mode": "error_passthrough",
            "error": str(error),
            "recovery_suggestions": ["Check LangGraph installation"]
        }

    # Node Functions
    def input_node(self, state: SimpleWorkflowState) -> SimpleWorkflowState:
        """First node: Receives input and initializes state"""
        with self.trace_operation("input_node", input=state.get("input_text", "")):
            state["timestamp"] = datetime.now().isoformat()
            state["step_count"] = state.get("step_count", 0) + 1
            return state

    def process_node(self, state: SimpleWorkflowState) -> SimpleWorkflowState:
        """Second node: Processes the input text"""
        with self.trace_operation("process_node", step=state["step_count"]):
            input_text = state.get("input_text", "")
            state["processed_text"] = input_text.upper()
            state["step_count"] += 1
            return state

    def output_node(self, state: SimpleWorkflowState) -> SimpleWorkflowState:
        """Final node: Generates output"""
        with self.trace_operation("output_node", step=state["step_count"]):
            output = f"[{state['timestamp']}] Processed: {state['processed_text']}"
            state["output_text"] = output
            state["step_count"] += 1
            return state

    def _create_graph(self) -> StateGraph:
        """Create the LangGraph workflow"""
        with self.trace_operation("create_graph"):
            # Create graph with state schema
            workflow = StateGraph(SimpleWorkflowState)

            # Add nodes
            workflow.add_node("input_node", self.input_node)
            workflow.add_node("process_node", self.process_node)
            workflow.add_node("output_node", self.output_node)

            # Add edges (defines flow)
            workflow.set_entry_point("input_node")
            workflow.add_edge("input_node", "process_node")
            workflow.add_edge("process_node", "output_node")
            workflow.add_edge("output_node", END)

            return workflow.compile()

    def run_workflow(self, input_text: str) -> Dict[str, Any]:
        """Execute the workflow with given input"""
        with self.trace_operation("run_workflow", input_length=len(input_text)):
            initial_state: SimpleWorkflowState = {
                "input_text": input_text,
                "processed_text": "",
                "output_text": "",
                "timestamp": "",
                "step_count": 0
            }
            return self.graph.invoke(initial_state)

print("✅ Orchestrator class defined")

## Step 3: Create and Run Workflow

Now let's create an instance and run some examples.

In [ ]:
# Create orchestrator
orchestrator = SimpleWorkflowOrchestrator()
print("✅ Orchestrator created")
print("📊 Check Jaeger UI: http://localhost:16686")

### Example 1: Basic Text Processing

In [ ]:
result1 = orchestrator.run_workflow("Hello, LangGraph!")

print(f"Input: {result1['input_text']}")
print(f"Output: {result1['output_text']}")
print(f"Steps: {result1['step_count']}")
print(f"Timestamp: {result1['timestamp']}")

### Example 2: Longer Text

In [ ]:
result2 = orchestrator.run_workflow("This is a simple workflow example demonstrating LangGraph")

print(f"Input: {result2['input_text']}")
print(f"Output: {result2['output_text']}")
print(f"Steps: {result2['step_count']}")

### Example 3: Multiple Runs

In [ ]:
test_inputs = [
    "Testing node execution",
    "State flow through graph",
    "ReflectiveModule integration"
]

print("Running multiple workflows:")
print("=" * 60)
for i, text in enumerate(test_inputs, 1):
    result = orchestrator.run_workflow(text)
    print(f"{i}. {result['output_text']}")

## Step 4: View Traces in Jaeger

Now check the Jaeger UI to see the traces:

1. Open http://localhost:16686
2. Select service: `beast-mode`
3. Look for operations:
   - `run_workflow`
   - `input_node`
   - `process_node`
   - `output_node`
4. Click on a trace to see the execution flow

You should see a clear visualization of how state flows through the graph!

## Key Learnings

1. **StateGraph**: Takes a TypedDict schema that defines state structure
2. **Nodes**: Simple functions with signature `state → state`
3. **Edges**: Define the workflow flow between nodes
4. **ReflectiveModule**: Gives Jaeger tracing automatically via `trace_operation()` context manager
5. **Compile**: Must compile the graph before running
6. **Invoke**: Use `graph.invoke(initial_state)` to execute

## Next Steps

- Example 2: Conditional routing and retry logic
- Example 3: Parallel agent execution
- Example 4: State persistence and checkpointing